In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Inisialisasi Qubit

<details>
<summary><b>Versi paket</b></summary>

Kode di halaman ini dikembangkan menggunakan persyaratan berikut.
Kami merekomendasikan menggunakan versi ini atau yang lebih baru.

```
qiskit-ibm-runtime~=0.43.1
```
</details>
Saat sebuah Circuit dieksekusi pada unit pemrosesan kuantum (QPU) IBM&reg;, biasanya dilakukan reset implisit di awal Circuit untuk memastikan qubit diinisialisasi ke nol. Ini dikendalikan oleh flag `init_qubits`, yang diatur sebagai [opsi eksekusi primitive](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-execution-options-v2).

Namun, proses reset tidak sempurna, sehingga menimbulkan kesalahan persiapan state. Untuk mengurangi kesalahan ini, sistem juga menyisipkan waktu tunda repetisi (atau `rep_delay`) di antara circuit. Setiap Backend memiliki `rep_delay` default yang berbeda, tapi biasanya lebih panjang dari T1 agar lingkungan dapat mereset qubit. `rep_delay` default bisa dikueri dengan menjalankan `backend.default_rep_delay`.

Semua QPU IBM menggunakan eksekusi repetition rate dinamis, yang memungkinkan kamu mengubah `rep_delay` untuk setiap job. Circuit yang kamu kirimkan dalam sebuah primitive job digabungkan untuk dieksekusi di QPU. Circuit-circuit ini dieksekusi dengan mengiterasi setiap circuit untuk setiap shot yang diminta; eksekusinya bersifat kolom-per-kolom pada matriks circuit dan shot, seperti yang diilustrasikan pada gambar berikut.

![Kolom pertama mewakili shot 0.  Circuit dijalankan secara berurutan dari 0 hingga 3.  Kolom kedua mewakili shot 1.  Circuit dijalankan secara berurutan dari 0 hingga 3.  Kolom-kolom berikutnya mengikuti pola yang sama. ](../docs/images/guides/repetition-rate-execution/circuits_shots_matrix1.avif "Matriks eksekusi kolom-per-kolom")

Karena `rep_delay` disisipkan di antara circuit, setiap shot eksekusi mengalami tunda ini. Oleh karena itu, menurunkan `rep_delay` akan mengurangi total waktu eksekusi QPU, namun dengan konsekuensi meningkatnya tingkat kesalahan persiapan state, seperti yang terlihat pada gambar berikut:

![Gambar ini menunjukkan bahwa semakin rendah nilai `rep_delay`, semakin tinggi tingkat kesalahan persiapan state.](../docs/images/guides/repetition-rate-execution/rep_delay.avif "Repetition delay versus tingkat kesalahan")

Menetapkan `rep_delay=0` dan `init_qubits=False` secara bersamaan pada dasarnya "menggabungkan" circuit-circuit tersebut, karena qubit akan dimulai dari state akhir shot sebelumnya.

Perlu diketahui bahwa meskipun circuit dalam sebuah primitive job digabungkan untuk eksekusi QPU, tidak ada jaminan urutan circuit dari PUB akan dieksekusi. Jadi, meskipun kamu mengirimkan `pubs=[pub1, pub2]`, tidak ada jaminan circuit dari `pub1` akan dijalankan sebelum `pub2`. Tidak ada pula jaminan bahwa circuit dari job yang sama akan dijalankan sebagai satu batch tunggal di QPU.

## Menentukan rep_delay untuk sebuah primitive job

In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler

service = QiskitRuntimeService()

# Make sure your backend supports it
backend = service.least_busy(
    operational=True, min_num_qubits=100, dynamic_reprate_enabled=True
)

# Determine the allowable range
backend.rep_delay_range
sampler = Sampler(mode=backend)

# Specify a value in the supported range
sampler.options.execution.rep_delay = 0.0005

## Langkah selanjutnya
> **Tip:** - Coba contoh di tutorial [Quantum approximate optimization algorithm (QAOA)](/tutorials/quantum-approximate-optimization-algorithm).
>   - Tinjau cara [memulai dengan primitives.](./get-started-with-primitives)